# Bonus: let the optimizer change more than the instructions

In Module 5 the optimizer could only rewrite TrailMate's instructions. That was on purpose: one lever at a time. Now let's hand it a second lever, the **model**, and see what it does with the choice.

**Business lens:** a smaller model that answers just as well is a real saving. Can the optimizer find one for us?
**Engineering lens:** the optimizer builds candidates from the attributes we allow (`target_attributes`) and the models we list (`model_search_space`), then scores each one with our rubric on `dev`.

| | Module 5 | This bonus |
|---|---|---|
| What can change | Instructions | Instructions **and** model |
| Models it can pick | `gpt-5.4` | `gpt-5.4`, `gpt-5.4-mini`, `mr-default` |
| Candidates | 3 | 3 |
| Promotion | Gated by evidence | Never. This is an experiment |

**Before we start:** finish notebook 05. We reuse its rubric, datasets and retained baseline. This bonus sits outside the 60-minute core: the instructions-only optimizer took about 13 minutes in our rehearsal, and a two-lever search may take longer. Run it after notebook 05 or 06, not while another notebook is waiting on a job, because they share `.lab-state.json`. It never changes the main agent.

In [1]:
import sys
import hashlib
from pathlib import Path

import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from the LAB663 repository.")
sys.path.insert(0, str(root / "src"))
import lab

project_client, openai_client = lab.connect()
test_cases = lab.load_cases("test")

state = lab.load_state()
if not {"rubric", "rubric_version", "evaluation_id", "test_dataset_id", "v2_version", "quality_baseline_test"} <= state.keys():
    raise RuntimeError("Finish notebook 05, including its promotion decision, before this bonus.")

# The baseline is whatever notebook 05 kept: the promoted candidate, or v1.
baseline = next(v for v in project_client.agents.list_versions(lab.AGENT_NAME) if str(v.version) == state["v2_version"])
dev_version = hashlib.sha256((lab.DATA / "dev.jsonl").read_bytes()).hexdigest()[:12]
dev_dataset = project_client.datasets.get(name="trailmate-dev", version=dev_version)

print(f"Baseline : {lab.AGENT_NAME} version {baseline.version} on {baseline.definition.model}")
print(f"Rubric   : {state['rubric']} version {state['rubric_version']}")
print(f"Training : {dev_dataset.name} version {dev_dataset.version}")

Baseline : trailmate version 2 on gpt-5.4
Rubric   : trailmate-rubric-5a76e5 version 1
Training : trailmate-dev version 01bd3d52951e


**Expected:** the retained baseline from notebook 05, our rubric and the `dev` dataset. If the cell stops, finish notebook 05 first.

## 1. What can the optimizer change?

An agent has several parts. The optimizer only touches the parts we list in `target_attributes`, and only if the agent actually has them.

In [2]:
from azure.ai.projects.models import TargetAttribute

print("Attributes the optimizer can change:", [a.value for a in TargetAttribute])
print("Tools on our baseline:", [getattr(t, "type", type(t).__name__) for t in baseline.definition.tools or []])

Attributes the optimizer can change: ['instructions', 'model', 'skills', 'tools']
Tools on our baseline: ['file_search']


**Expected:** four attributes (`instructions`, `model`, `skills`, `tools`) and one tool, `file_search`.

**Read the output.** Two of the four don't apply to TrailMate:

| Attribute | What the optimizer does | For TrailMate |
|---|---|---|
| `instructions` | Rewrites the system prompt | ✅ We used this in Module 5 |
| `model` | Picks a model or deployment from our list | ✅ New in this bonus |
| `tools` | Rewrites **function** tool and parameter descriptions; names and schemas stay | ❌ File search isn't a function tool, so there's nothing to rewrite |
| `skills` | Rewrites skill descriptions and bodies | ❌ TrailMate has no skills |

So we give it two levers: instructions and model. The optimizer can only improve what the agent exposes.

## 2. Which models may it try?

We list deployments that already exist in our project. The baseline's `gpt-5.4` is always in the running.

**Predict:** will the best candidate stay on `gpt-5.4`, move to `gpt-5.4-mini`, or move to the router?

Two things to know before we look:

- **The goal is quality only.** `improve_quality` is the only optimization goal, so a smaller model has to score *at least as well* to win. The optimizer won't prefer it for being cheaper; cost is still our call.
- **The router may stumble.** In Module 6 the router plus file search returned HTTP 500 on some questions. A router candidate may come back `failed` or score lower. That's a reliability signal, not our mistake.

In [3]:
from azure.ai.projects.models import (
    AgentOptimizationCandidateSearchConfiguration,
    AgentOptimizationConfiguration,
    AgentOptimizationEstimateInputs,
    AgentOptimizationEvaluationConfiguration,
    AgentOptimizationEvaluator,
    AgentOptimizationFoundryAgentTargetConfiguration,
    AgentOptimizationJob,
    AgentOptimizationModelConfiguration,
    AgentOptimizationSpace,
    AgentOptimizationTargetCompletionDatasetReferenceDataSource,
    AgentOptimizationTargetCompletionEvaluationSet,
    EvaluationModelConfiguration,
)

EXPERIMENT = "bonus"  # Change this name to start a separate experiment; saved IDs are kept per name.
TARGET_ATTRIBUTES = ["instructions", "model"]
MODEL_CHOICES = ["gpt-5.4-mini", "mr-default"]
MAX_CANDIDATES = 3

deployed = {d.name for d in project_client.deployments.list()}
missing = [m for m in MODEL_CHOICES if m not in deployed]
if missing:
    raise RuntimeError(f"These deployments don't exist in our project: {missing}")

target = AgentOptimizationFoundryAgentTargetConfiguration(name=lab.AGENT_NAME, version=str(baseline.version))
writer = AgentOptimizationModelConfiguration(model=lab.FRONTIER_MODEL)  # the model that proposes changes

settings = AgentOptimizationConfiguration(
    evaluation_configuration=AgentOptimizationEvaluationConfiguration(
        training_set=AgentOptimizationTargetCompletionEvaluationSet(
            source=AgentOptimizationTargetCompletionDatasetReferenceDataSource(name=dev_dataset.name, version=dev_dataset.version)
        ),
        evaluators=[AgentOptimizationEvaluator(name=state["rubric"], version=state["rubric_version"])],
        evaluation_model=EvaluationModelConfiguration(model=lab.FRONTIER_MODEL),  # same judge as Module 5
    ),
    candidate_search_configuration=AgentOptimizationCandidateSearchConfiguration(max_candidates=MAX_CANDIDATES),
    agent_optimization_space=AgentOptimizationSpace(
        target_attributes=TARGET_ATTRIBUTES,  # two levers this time
        model_search_space=MODEL_CHOICES,
    ),
)

estimate = project_client.agents.estimate_optimization_job(
    AgentOptimizationEstimateInputs(
        optimization_model_configuration=writer, optimization_configuration=settings, target_configuration=target
    )
)
print(estimate)

{'call_counts': {'agent': {'low': 48.0, 'typical': 62.099999999999994, 'ceiling': 207.0}, 'evaluation': {'low': 48.0, 'typical': 62.099999999999994, 'ceiling': 207.0}, 'optimization': {'low': 0.0, 'typical': 4.5, 'ceiling': 15.0}}, 'cost': {'currency': 'USD', 'total': {'low': 0.9743999999999999, 'typical': 1.4631299999999998, 'ceiling': 4.8770999999999995}, 'by_stage': {'agent': {'low': 0.528, 'typical': 0.6830999999999999, 'ceiling': 2.2769999999999997}, 'evaluation': {'low': 0.44639999999999996, 'typical': 0.5775299999999999, 'ceiling': 1.9250999999999998}, 'optimization': {'low': 0.0, 'typical': 0.20249999999999999, 'ceiling': 0.6749999999999999}}}, 'prices_as_of': 1791158400}


**Expected:** an estimate with call counts and a cost range. Compare it with Module 5's estimate: does a second lever cost more to search?

## 3. Let's start the search

We start the job once. Its ID is saved under our `EXPERIMENT` name, so rerunning this cell resumes the same job instead of starting another. While it runs, try a challenge in [the wait-time notebook](wait-time-challenges.ipynb).

In [ ]:
job_key, context_key = f"{EXPERIMENT}_optimization_job_id", f"{EXPERIMENT}_optimization_context"
job_context = {"baseline": str(baseline.version), "rubric": state["rubric"], "version": state["rubric_version"],
               "settings": settings.as_dict(), "writer": writer.as_dict()}

state = lab.load_state()
if job_key in state:
    if state[context_key] != job_context:
        raise RuntimeError(f"Experiment '{EXPERIMENT}' was saved with other settings. Use a new EXPERIMENT name to try these.")
    job_id = state[job_key]
else:
    poller = project_client.agents.begin_create_optimization_job(
        AgentOptimizationJob(target_configuration=target, optimization_model_configuration=writer,
                             optimization_configuration=settings, display_name=f"TrailMate {EXPERIMENT}: instructions and model")
    )
    job_id = poller.details["job_id"]
    lab.save_state(**{job_key: job_id, context_key: job_context})

print("Optimization job:", job_id)
result = lab.wait_for_optimization_job(project_client, job_id)
print("Finished. Best score:", result.candidate_summary.best_score, "| baseline:", result.candidate_summary.baseline_score)

Optimization job: opt_3ebb49c264fa4810912b5a2ad260a778
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (0s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (15s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (31s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (46s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (61s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (76s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (92s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (107s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a778: in_progress (122s). Try a local wait-time challenge.
Optimizer opt_3ebb49c264fa4810912b5a2ad260a7

**Expected:** progress lines, then the best score and the baseline score from the optimizer's own `dev` run. If the wait times out, rerun the cell: it resumes the saved job.

## 4. What did each candidate change?

This is the interesting part. Each candidate is a set of **mutations** on the baseline: a new model, new instructions, or both.

In [ ]:
def value(x):
    return getattr(x, "value", x)

candidates = list(project_client.agents.list_optimization_candidates(job_id, expand=["mutations"]))

rows = []
for c in candidates:
    changes = {value(m.type): m.value for m in (c.output.mutations if c.output else [])}
    rows.append({
        "candidate": c.name,
        "status": value(c.status),
        "model": changes.get("model", baseline.definition.model),
        "new instructions": "instructions" in changes,
        "score": c.evaluation.score if c.evaluation else None,
        "avg_tokens": c.evaluation.avg_tokens if c.evaluation else None,
        "_instructions": changes.get("instructions", baseline.definition.instructions),
        "_id": c.candidate_id,
    })

board = pd.DataFrame(rows)
board.drop(columns=["_instructions", "_id"])

**Expected:** the baseline plus up to three candidates, each with a model, whether its instructions changed, a score and average tokens.

**Read the table by column.**

- **Model:** did any candidate leave `gpt-5.4`? Which one?
- **New instructions:** did a model switch come with a rewrite? A smaller model sometimes needs firmer instructions to hold quality.
- **Score and tokens:** a candidate that scores about the same with fewer tokens is worth a closer look, even if it isn't the "best".
- **Status:** a `failed` router candidate is evidence about reliability. The score can't show it.

**Think:** the optimizer ranks by score alone. Would *you* pick the top scorer, or a slightly lower score on a smaller model?

In [ ]:
done = board[(board["status"] == "completed") & board["score"].notna() & (board["candidate"] != "baseline")]
if done.empty:
    raise RuntimeError("No completed candidate. Read the statuses above and keep the baseline.")
best = done.sort_values(["score", "candidate"], ascending=False).iloc[0]

print(f"Best candidate : {best['candidate']}")
print(f"Model          : {best['model']}")
print(f"Score on dev   : {best['score']:.3f}")
print(f"New instructions: {best['new instructions']}")
if best["new instructions"]:
    print("\n" + best["_instructions"])

**Expected:** the best candidate's model and score, and its instructions if it rewrote them.

**Read the output.** As in Module 5, check the instructions with a human eye. Did it copy product facts into the prompt? Did it add a rule for off-topic requests, the one gap Module 5's rewrite missed?

## 5. Does it hold on questions it never saw?

The score above comes from `dev`, which the optimizer trained on. Let's stage the best candidate as a **separate agent** and score it on `test`, then compare it with the baseline's saved test score from notebook 05. Nothing is promoted.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

if best["model"] not in deployed:
    raise RuntimeError(f"The optimizer picked '{best['model']}', which isn't a deployment in our project. Stop here.")

bonus_name = f"{lab.AGENT_NAME}-{EXPERIMENT}-candidate"
definition = PromptAgentDefinition(model=best["model"], instructions=best["_instructions"], tools=baseline.definition.tools)

state = lab.load_state()
if state.get(f"{EXPERIMENT}_candidate_id") == best["_id"]:
    staged = next(v for v in project_client.agents.list_versions(bonus_name)
                  if str(v.version) == state[f"{EXPERIMENT}_candidate_version"])
else:
    staged = project_client.agents.create_version(bonus_name, definition=definition)
    lab.save_state(**{f"{EXPERIMENT}_candidate_id": best["_id"], f"{EXPERIMENT}_candidate_version": str(staged.version)})

if staged.definition.as_dict() != definition.as_dict():
    raise RuntimeError("The staged definition differs from the selected candidate. Don't evaluate it.")
print(f"Staged {bonus_name}, version {staged.version}, on {staged.definition.model}. The main agent is unchanged.")

In [ ]:
bonus_test = lab.run_repeated_eval(openai_client, state["evaluation_id"], state["test_dataset_id"], bonus_name,
                                   staged.version, f"{EXPERIMENT} candidate on test", cases=test_cases)
kept = state["quality_baseline_test"]

display(pd.DataFrame([
    {"version": "retained baseline", "model": baseline.definition.model,
     **{k: kept[k] for k in ("score", "spread", "tokens", "errored")}},
    {"version": f"{EXPERIMENT} candidate", "model": staged.definition.model,
     **{k: bonus_test[k] for k in ("score", "spread", "tokens", "errored")}},
]).round(3))

if bonus_test["valid"]:
    print(pd.DataFrame({
        "retained baseline": lab.scores_by_way(kept["per_item"], test_cases),
        f"{EXPERIMENT} candidate": lab.scores_by_way(bonus_test["per_item"], test_cases),
    }).round(3))
else:
    print("Incomplete evaluation: there's no comparable score. A router candidate may hit the known router + file search error.")

**Expected:** two rows, the retained baseline and our candidate, with scores on `test`. If both are valid, a table by way of losing trust follows. If the candidate is on the router, the batch may be incomplete: that's an honest "we can't compare", not a low score.

**Read the output.** Compare the gap with the spreads, as in Module 5. Then compare tokens: if the candidate is on a smaller model and holds quality, that's the saving. To put a dollar figure on it we'd still need verified prices in `data/pricing.json`.

## 6. So what did the second lever teach us? (the aha)

> **Aha:** The optimizer searches only what we let it touch, and judges only by the goal we give it. With a quality-only goal, a smaller model must *earn* its place on score; the cost decision stays with us. And it can only optimize what the agent exposes: no function tools, nothing for `tools` to rewrite.

We also took away:

- **More levers, bigger search.** Each attribute multiplies the combinations, so a fixed number of candidates covers less of the space.
- **A model switch and a rewrite go together.** A smaller model often needs different instructions to hold quality.
- **Reliability is ours to check.** A failed or incomplete router candidate is evidence the score alone can't show.
- **Nothing was promoted.** The main agent and notebook 06's state are unchanged.

**Go further (optional):** set `EXPERIMENT = "model-only"` and `TARGET_ATTRIBUTES = ["model"]`, then run sections 2 to 5 again. With the instructions frozen, does `gpt-5.4-mini` hold up on its own? Check the estimate first.